<a href="https://colab.research.google.com/github/pejmanrasti/Big_Data_M2/blob/main/lab1_storage_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 · Parquet & DuckDB, measured — **solution**
**Big Data at Scale · Session 1 · M2 Data Science**

In this lab you measure the storage tricks from Part 1 on real data: 6 months of New York yellow-taxi trips (2024, about 20 million rows).

| Step | What you do | Slide idea |
|---|---|---|
| 1 | Load the data as CSV in pandas: 1 month, then 6. Watch the RAM. | “Big” = it no longer fits your tool |
| 2 | Write the same data to Parquet. Compare sizes. Look inside the file. | Parquet anatomy, encoding |
| 3 | Query CSV and Parquet with DuckDB: one column vs all columns. | Column pruning |
| 4 | Add a date filter. Then partition by month and repeat. | Predicate pushdown, partition pruning |
| 5 | Fill in the results table and explain your best result. | — |

> **Before you start:** *Runtime → Change runtime type → CPU* is enough. Run the cells from top to bottom. Every result you measure goes into the `results` dictionary used in step 5.

## 0 · Setup

In [ ]:
!pip install -q duckdb psutil

import os, time, gc, glob, urllib.request
import duckdb, psutil
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

proc = psutil.Process()

def ram_gb():
    "Memory used by this notebook (GB)."
    return proc.memory_info().rss / 1e9

def size_gb(path):
    "Size of a file or of a whole folder (GB)."
    if os.path.isfile(path):
        return os.path.getsize(path) / 1e9
    return sum(os.path.getsize(os.path.join(root, f))
               for root, _, files in os.walk(path) for f in files) / 1e9

def bench(sql, runs=3):
    "Run a DuckDB query several times and keep the best time (seconds)."
    best = float("inf")
    for _ in range(runs):
        t = time.perf_counter()
        con.execute(sql).fetchall()
        best = min(best, time.perf_counter() - t)
    return best

con = duckdb.connect()
results = {}

print(f"DuckDB {duckdb.__version__} · pandas {pd.__version__}")
print(f"This machine: {os.cpu_count()} cores, {psutil.virtual_memory().total/1e9:.1f} GB of RAM")

### Get the data
New York publishes the trip records as monthly **Parquet** files. To play the classic “raw CSV” scenario, we first turn them into CSV files ourselves.

In [ ]:
MONTHS = [f"2024-{m:02d}" for m in range(1, 7)]   # January to June 2024
URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{}.parquet"

os.makedirs("raw", exist_ok=True)
for m in MONTHS:
    path = f"raw/yellow_{m}.parquet"
    if not os.path.exists(path):
        urllib.request.urlretrieve(URL.format(m), path)
    print(f"{path}: {size_gb(path)*1000:.0f} MB")

In [ ]:
os.makedirs("csv", exist_ok=True)
for m in MONTHS:
    out = f"csv/yellow_{m}.csv"
    if not os.path.exists(out):
        con.execute(f"COPY (SELECT * FROM 'raw/yellow_{m}.parquet') TO '{out}' (HEADER)")
print(f"CSV folder: {size_gb('csv'):.2f} GB for {len(MONTHS)} months")
con.execute("SELECT * FROM 'csv/yellow_2024-01.csv' LIMIT 5").df()

## 1 · pandas on CSV: watch the RAM
pandas loads **the whole file into memory**, every column, every row. Let’s see what that costs.

In [ ]:
gc.collect(); before = ram_gb()
t = time.perf_counter()
df1 = pd.read_csv("csv/yellow_2024-01.csv")
load_1 = time.perf_counter() - t
ram_1 = ram_gb() - before

print(f"1 month: {len(df1):,} rows · loaded in {load_1:.1f} s · RAM used: +{ram_1:.2f} GB")
print(f"CSV file on disk: {size_gb('csv/yellow_2024-01.csv'):.2f} GB")
df1.dtypes

**Look at the types.** The two date columns are text (`object` or `str`): a CSV has no types, so pandas guessed. Text takes much more memory than numbers.

Now 6 months. Reading several files and concatenating them needs about **twice** the final memory for a moment. The cell below checks first whether this machine can afford it, and loads fewer months if not.

In [ ]:
del df1; gc.collect()

avail = psutil.virtual_memory().available / 1e9
need = ram_1 * len(MONTHS) * 2.2           # 6 months, ×2 while concatenating
n = len(MONTHS) if need < 0.8 * avail else max(1, int(0.8 * avail / (ram_1 * 2.2)))
print(f"6 months would need about {need:.1f} GB; {avail:.1f} GB available → loading {n} month(s)")

before = ram_gb()
t = time.perf_counter()
df6 = pd.concat([pd.read_csv(f"csv/yellow_{m}.csv") for m in MONTHS[:n]], ignore_index=True)
load_n = time.perf_counter() - t
ram_n = ram_gb() - before
print(f"{n} month(s): {len(df6):,} rows · loaded in {load_n:.1f} s · RAM used: +{ram_n:.2f} GB")

t = time.perf_counter(); avg_fare = df6["fare_amount"].mean(); calc = time.perf_counter() - t
print(f"Average fare: {avg_fare:.2f} $ · computing it took {calc*1000:.0f} ms (loading took {load_n:.0f} s)")

per_month = ram_n / n
print(f"\nAt {per_month:.2f} GB per month, 5 years of taxi data would need about "
      f"{per_month*60:.0f} GB of RAM. This machine has {psutil.virtual_memory().total/1e9:.0f} GB.")
results["pandas_load_s"] = load_n
results["pandas_months"] = n
del df6; gc.collect();

**What we learn:** the computation is fast; **loading** is the problem. pandas reads every byte of every column, keeps it all in RAM, and the RAM runs out long before “big data” sizes.

## 2 · Same data, written as Parquet
DuckDB can read all the CSV files and write one Parquet file **without loading everything into memory**: it streams the data. We also sort the rows by pickup time while writing; step 4 shows why.

In [ ]:
t = time.perf_counter()
con.execute('''
    COPY (SELECT * FROM read_csv('csv/*.csv', union_by_name = true)
          ORDER BY tpep_pickup_datetime)
    TO 'trips.parquet' (FORMAT parquet, COMPRESSION zstd)
''')
print(f"Written in {time.perf_counter() - t:.1f} s")

csv_gb, pq_gb = size_gb("csv"), size_gb("trips.parquet")
print(f"CSV:     {csv_gb:.2f} GB")
print(f"Parquet: {pq_gb:.2f} GB  →  {csv_gb/pq_gb:.1f} × smaller")
results["size_csv"], results["size_pq"] = csv_gb, pq_gb

### Look inside the Parquet file
Remember the slide *“A Parquet file carries its own map”*: row groups, column chunks and a footer with statistics. We can read that footer directly.

In [ ]:
meta = pq.ParquetFile("trips.parquet").metadata
print(f"{meta.num_rows:,} rows · {meta.num_columns} columns · {meta.num_row_groups} row groups")

# How well does each column compress?
names = [meta.schema.column(i).name for i in range(meta.num_columns)]
comp = [sum(meta.row_group(r).column(i).total_compressed_size for r in range(meta.num_row_groups)) for i in range(meta.num_columns)]
raw  = [sum(meta.row_group(r).column(i).total_uncompressed_size for r in range(meta.num_row_groups)) for i in range(meta.num_columns)]
cols = pd.DataFrame({"column": names, "stored MB": [c/1e6 for c in comp],
                     "uncompressed MB": [u/1e6 for u in raw]})
cols["× smaller"] = cols["uncompressed MB"] / cols["stored MB"]
cols.sort_values("stored MB", ascending=False).round(1)

Columns with few distinct values (`VendorID`, `payment_type`, `store_and_fwd_flag`…) shrink the most: dictionary and run-length encoding at work. Unique values like timestamps shrink the least.

Now the **statistics** of the pickup time in the first row groups:

In [ ]:
i_pick = names.index("tpep_pickup_datetime")
stats = []
for r in range(meta.num_row_groups):
    s = meta.row_group(r).column(i_pick).statistics
    stats.append({"row group": r, "rows": meta.row_group(r).num_rows, "min pickup": s.min, "max pickup": s.max})
stats = pd.DataFrame(stats)
stats.head(8)

## 3 · DuckDB queries: one column vs all columns
Same question asked to the CSV files and to the Parquet file. Each query runs a few times and we keep the best time.
* **one column:** the average fare → needs only `fare_amount`
* **all columns:** a checksum (hash) of every value of every column → must read everything

In [ ]:
SRC = {
    "CSV":     "read_csv('csv/*.csv', union_by_name = true)",
    "Parquet": "'trips.parquet'",
}
Q_ONE = "SELECT avg(fare_amount) FROM {src}"
Q_ALL = "SELECT sum(hash(COLUMNS(*))) FROM {src}"

for name, src in SRC.items():
    runs = 2 if name == "CSV" else 3
    results[f"{name}_one"] = bench(Q_ONE.format(src=src), runs)
    results[f"{name}_all"] = bench(Q_ALL.format(src=src), runs)
    print(f"{name:8s} one column: {results[f'{name}_one']:6.2f} s   all columns: {results[f'{name}_all']:6.2f} s")

print(f"\nCSV: reading one column is barely faster than reading all of them.")
print(f"Parquet: one column is {results['Parquet_all']/results['Parquet_one']:.0f} × faster than all columns.")

**Why?** A CSV file is stored row by row: to find the fares, DuckDB must still read and split every line. Parquet stores each column separately, so DuckDB reads only the `fare_amount` chunks. That is **column pruning**.

## 4 · Filters: statistics and partitions
### 4a · A date filter
Average fare for one day only: 15 March 2024.

In [ ]:
DAY = "tpep_pickup_datetime >= '2024-03-15' AND tpep_pickup_datetime < '2024-03-16'"
Q_FILTER = "SELECT count(*), avg(fare_amount) FROM {src} WHERE " + DAY

for name, src in SRC.items():
    runs = 2 if name == "CSV" else 3
    results[f"{name}_filter"] = bench(Q_FILTER.format(src=src), runs)
    print(f"{name:8s} with filter: {results[f'{name}_filter']:6.2f} s   (one column, no filter: {results[f'{name}_one']:.2f} s)")

# How many row groups could contain 15 March? The footer statistics tell us.
day_lo, day_hi = pd.Timestamp("2024-03-15"), pd.Timestamp("2024-03-16")
can_match = stats[(pd.to_datetime(stats["max pickup"]) >= day_lo) & (pd.to_datetime(stats["min pickup"]) < day_hi)]
print(f"\nRow groups that can contain 15 March: {len(can_match)} of {len(stats)}. DuckDB skips the others.")

**Predicate pushdown:** DuckDB compares the date with the min/max of each row group and skips groups that cannot match. The CSV gets no such help: every line must be read.

### Bonus · Why sorting matters
We sorted the Parquet file by pickup time. What if the rows were in random order instead? *(Takes a minute or two.)*

In [ ]:
con.execute('''
    COPY (SELECT * FROM 'trips.parquet' ORDER BY random())
    TO 'trips_shuffled.parquet' (FORMAT parquet, COMPRESSION zstd)
''')
m2 = pq.ParquetFile("trips_shuffled.parquet").metadata
s2 = [m2.row_group(r).column(i_pick).statistics for r in range(m2.num_row_groups)]
hit = sum(1 for s in s2 if pd.Timestamp(s.max) >= day_lo and pd.Timestamp(s.min) < day_hi)

t_sorted = results["Parquet_filter"]
t_shuffled = bench(Q_FILTER.format(src="'trips_shuffled.parquet'"))
print(f"Shuffled file: {size_gb('trips_shuffled.parquet'):.2f} GB (sorted: {size_gb('trips.parquet'):.2f} GB)")
print(f"Row groups that can contain 15 March: {hit} of {m2.num_row_groups}")
print(f"Filter query: {t_shuffled:.2f} s shuffled vs {t_sorted:.2f} s sorted")

Shuffled, every row group covers January to June, so **no row group can be skipped**. The file is also bigger: random order breaks the runs of repeated values. Sorting (or clustering) data on the columns you filter on is a real design decision.

### 4b · Partition by month
Now we write one folder per month. Note the `WHERE`: a few trips have impossible dates (data quality!) and we leave them out.

In [ ]:
con.execute('''
    COPY (
        SELECT *, year(tpep_pickup_datetime) AS year, month(tpep_pickup_datetime) AS month
        FROM 'trips.parquet'
        WHERE tpep_pickup_datetime >= '2024-01-01' AND tpep_pickup_datetime < '2024-07-01'
    )
    TO 'trips_by_month' (FORMAT parquet, PARTITION_BY (year, month), COMPRESSION zstd, OVERWRITE_OR_IGNORE)
''')

outside = con.execute("SELECT count(*) FROM 'trips.parquet' WHERE tpep_pickup_datetime < '2024-01-01' "
                  "OR tpep_pickup_datetime >= '2024-07-01'").fetchone()[0]
print(f"Trips with a pickup date outside January–June 2024: {outside:,}\n")

for root, dirs, files in sorted(os.walk("trips_by_month")):
    depth = root.count(os.sep)
    print("    " * depth + os.path.basename(root) + "/", f"({len(files)} file)" if files else "")
results["size_part"] = size_gb("trips_by_month")

In [ ]:
SRC["Parquet, partitioned by month"] = "read_parquet('trips_by_month/*/*/*.parquet', hive_partitioning = true)"
src = SRC["Parquet, partitioned by month"]
Q_FILTER_PART = "SELECT count(*), avg(fare_amount) FROM {src} WHERE year = 2024 AND month = 3 AND " + DAY

results["Part_one"] = bench(Q_ONE.format(src=src))
results["Part_all"] = bench(Q_ALL.format(src=src))
results["Part_filter"] = bench(Q_FILTER_PART.format(src=src))
print(f"Partitioned · one column: {results['Part_one']:.2f} s · all columns: {results['Part_all']:.2f} s · "
      f"with filter: {results['Part_filter']:.2f} s")
print(f"The filter opens 1 folder out of {len(glob.glob('trips_by_month/*/*'))}.")

**Partition pruning:** the year and month are in the folder names, so DuckDB skips 5 folders out of 6 without even opening the files. Then the row-group statistics do the rest inside the March file.

## 5 · Results

In [ ]:
table = pd.DataFrame(
    {
        "Size on disk (GB)": [results["size_csv"], results["size_pq"], results["size_part"]],
        "1 column (s)":      [results["CSV_one"], results["Parquet_one"], results["Part_one"]],
        "All columns (s)":   [results["CSV_all"], results["Parquet_all"], results["Part_all"]],
        "With filter (s)":   [results["CSV_filter"], results["Parquet_filter"], results["Part_filter"]],
    },
    index=["CSV", "Parquet", "Parquet, partitioned by month"],
)
display(table.round(3))
print("Speed-up compared with CSV:")
display((table.iloc[0, 1:] / table.iloc[:, 1:]).round(1).astype(str) + " ×")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
table.iloc[:, 1:].T.plot.bar(ax=ax, logy=True, rot=0, color=["#C2571A", "#2D6A9F", "#14213D"])
ax.set_ylabel("seconds (log scale)")
ax.set_title("Same questions, three storage layouts")
plt.tight_layout(); plt.show()

## Questions and answers

**1. Why is the CSV so much bigger than the Parquet file?**
CSV stores every value as text, with no types, no encoding and no compression. Parquet stores typed values column by column, encodes repeated values (dictionary, run-length) and compresses each column chunk.

**2. Why is “one column” barely faster than “all columns” on CSV?**
A CSV is stored row by row. To get one field, the reader must still read and split every line. Parquet keeps each column apart, so it reads only what the query needs (**column pruning**).

**3. Why does the date filter help on Parquet but not on CSV?**
Parquet’s footer stores the min/max of each row group. The engine skips row groups whose range cannot contain the date (**predicate pushdown**). A CSV has no statistics: every line must be read and checked.

**4. Why did the shuffled file lose that advantage?**
In random order every row group covers the whole period, so no group can be ruled out. Statistics only help when the data is sorted or clustered on the filter column. The shuffled file is also bigger because random order breaks the runs of repeated values.

**5. What does partitioning add?**
The filter values are in the folder names, so whole folders are skipped before any file is opened (**partition pruning**). Partition on columns you filter on and that have few values (date, country), not on IDs: millions of tiny folders create the small-files problem.

**6. Why could DuckDB handle 6 months when pandas struggled?**
pandas loads all the data into memory before computing. DuckDB streams the files chunk by chunk and reads only the columns and row groups it needs, so memory use stays small. Reading less is the first scaling technique, before any cluster.